# Partie 2 - Mini-projet : [TITRE DE VOTRE PROJET]

**Binôme** : [Nom 1] - [Nom 2]
**Jeu de données** : [nom + lien UCI / Kaggle]
**Problème** : [régression ou classification] - variable cible : `[cible]`

Barème : prétraitement (25 %) - comparaison des modèles (30 %) - interprétation (30 %) - clarté (15 %).

## 1. Présentation du problème et des données
Décrire en quelques lignes : contexte, taille du jeu de données, variables, cible, intérêt métier.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

DATA_PATH = "../data/votre_fichier.csv"   # <-- à modifier
TARGET = "cible"                          # <-- à modifier
TASK = "classification"                   # "classification" ou "regression"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 2. Exploration (EDA)

In [ ]:
df.info()
print(df.isnull().sum())
df.describe(include="all")

## 3. Prétraitement (25 %)
Valeurs manquantes, outliers, encodage, mise à l'échelle - **justifier chaque choix**.

In [ ]:
# df = df.dropna()
# df = pd.get_dummies(df, drop_first=True)
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Comparaison rigoureuse des modèles (30 %)
Au minimum 3 modèles, même découpage, validation croisée, mêmes métriques.

In [ ]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

if TASK == "classification":
    models = {
        "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier()),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42),
    }
    scoring = "f1_macro"
else:
    models = {
        "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
        "KNN": make_pipeline(StandardScaler(), KNeighborsRegressor()),
        "Decision Tree": DecisionTreeRegressor(random_state=42),
        "Random Forest": RandomForestRegressor(random_state=42),
    }
    scoring = "r2"

rows = []
for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring=scoring)
    model.fit(X_train, y_train)
    rows.append({"Modèle": name, f"CV {scoring} (moy)": cv.mean(), "CV std": cv.std(), "Test score": model.score(X_test, y_test)})
pd.DataFrame(rows).set_index("Modèle").round(3)

## 5. Interprétation des résultats (30 %)
Matrice de confusion / résidus, importance des variables, sur/sous-apprentissage (écart train vs test).

In [ ]:
# Exemple : importance des variables du Random Forest
rf = models["Random Forest"]
pd.Series(rf.feature_importances_, index=X.columns).sort_values().tail(15).plot.barh(title="Importance des variables")
plt.show()

## 6. Conclusion critique
- **Modèle retenu** : ...
- **Pourquoi** : performances, stabilité (CV), interprétabilité, coût de calcul.
- **Limites** : ...
- **Améliorations possibles** : ...